# Repeated full PEC: several transpiled circuits × repeats (reproducibility vs noise)

Runs full PEC on one device for **N_CIRCUITS transpiled circuits** (fixed transpiler seeds, which place the
circuit on different qubits), each evaluated **N_REPEATS times**. Every run is an **independent job** (own
noise learning + own sampling). All jobs are submitted together in one Batch, so the group queues once.

* Within one circuit (same ISA circuit, reused): run-to-run reproducibility of the device.
* Across circuits (different qubits, different summed CZ error): dependence on the noise of the gates used.

Per run the notebook records the result and the noise measures: the summed reported CZ error of the gates
used, from the calibration in effect when the job started (`target_history`), and the noise learned by PEC
in that job (sampling overhead and per-layer overheads from the metadata).

PEC settings identical to `PEC_single_step.ipynb` (MODE "integrated"): TREX + PEC, noise learned for all
distinct layers, no overhead limit, default precision 1/64.

**Circuits.** Transpiled once at submission (Cell 4), on the calibration of that moment, with the same rule as
before: optimization level 1, CZ couplers with reported error > 1% excluded. Seeds are taken from
`FIRST_SEED` upward; a seed whose circuit is identical to an earlier one is skipped, until `N_CIRCUITS`
distinct circuits are found (no selection by quality). Each circuit is saved as QPY with its observables and
reused for all its repeats; a summary of all circuits is saved as `pec_repeats_circuits_<backend>_<label>.json`
(no job ids).

**Run order:** 1–3 → 4 (transpile + checks, no QPU) → 4b (circuit drawings, no QPU) → 5 (estimate and budget,
no QPU) → 6 (submit; returns immediately) → 7 (collect; re-run until all DONE; safe after a kernel restart:
run 1–3, then 7) → 8 (analysis).

In [ ]:
# ── Cell 1: settings ──
from datetime import datetime, timezone
DRY_RUN       = False
SERVICE_NAME  = "NTU"
INSTANCE      = "<YOUR_INSTANCE_NAME>"
BACKEND_NAME  = "ibm_phoenix"
N_CIRCUITS    = 4             # distinct transpiled circuits
N_REPEATS     = 3             # independent PEC jobs per circuit
FIRST_SEED    = 0             # seeds tried from here upward (identical circuits skipped)
MAX_SEED      = 30            # stop searching after this seed
CZ_ERR_MAX    = 0.01          # exclude CZ couplers with reported error above this
EXEC_MODE     = "batch"       # "batch" or "session"
GRP_ID, REP   = 0, 0
QPU_BUDGET_S  = 94 * 60       # QPU time available on this instance
SEC_PER_RUN   = 70            # planning value at overhead ~10 (9 Oct phoenix: 64 s)
MAX_EXEC_S    = 900           # cap per job (a capped job returns nothing and is still charged)
PEC_PRECISION = None          # None = default 1/64
LN_FIT        = (32.1, -1.03, 0.4)   # ln(overhead) ≈ a·ΣCZ + b, |residual| ≤ 0.4 (4 earlier runs, 3 devices)
INCLUDE_PREVIOUS = ["pec_result_ibm_phoenix_seed0_excl1pct.json"]   # 9 Oct single run (shown separately)

In [ ]:
# ── Cell 2: imports + circuit (from Comparsion.ipynb) ──
"""
OTC-based Observable Comparison Experiment
比較 SV, Noise Model, FakeMarrakesh, ibm_marrakesh (Baseline/ZNE/Twirling/ZNE+Twirling/QESEM)
Group 1 (target=+0.8): Z0, Z4, Z6, X5, X7
Group 2 (target=-0.8): Z1, Z5, Z7, X4, X6
"""

import numpy as np
import pandas as pd
import csv, time
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.primitives import StatevectorEstimator
from qiskit.quantum_info import SparsePauliOp
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel
from qiskit_ibm_runtime.fake_provider import FakeMarrakesh
from qiskit_ibm_runtime import QiskitRuntimeService, Session
from qiskit_ibm_runtime import EstimatorV2 as Estimator
from qiskit_ibm_runtime.options import EstimatorOptions
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

import os, json, glob
from qiskit import qpy
from qiskit.quantum_info import PauliList
from qiskit_ibm_runtime import Batch, Session
from qiskit.transpiler import Target

def build_qru_circuit(data_input, hidden_input, params):
    """
    data_input:   numpy array, shape (8,)
    hidden_input: numpy array, shape (8,), 初始為全零
    params:       numpy array, shape (131,)
    """

    data   = QuantumRegister(4, 'data')
    hidden = QuantumRegister(4, 'hidden')
    anc    = QuantumRegister(2, 'anc')
    qc = QuantumCircuit(data, hidden, anc)

    # ==================
    # 1. ENCODING
    # ==================
    # Data: params[0:16], 每 qubit 4 個參數
    qc.barrier(label='ENC')
    for i in range(4):
        rx_angle = data_input[2*i] * params[4*i]     + params[4*i + 1]
        ry_angle = data_input[2*i + 1] * params[4*i + 2] + params[4*i + 3]
        qc.rx(rx_angle, data[i])
        qc.ry(ry_angle, data[i])

    # Hidden: params[16:32]
    for i in range(4):
        rx_angle = hidden_input[2*i] * params[16 + 4*i]     + params[16 + 4*i + 1]
        ry_angle = hidden_input[2*i + 1] * params[16 + 4*i + 2] + params[16 + 4*i + 3]
        qc.rx(rx_angle, hidden[i])
        qc.ry(ry_angle, hidden[i])

    # 循環 CX on data: 0→1→2→3→0
    for i in range(4):
        qc.cx(data[i], data[(i+1) % 4])

    # 循環 CX on hidden: 4→5→6→7→4
    for i in range(4):
        qc.cx(hidden[i], hidden[(i+1) % 4])

    # ==================
    # 2. UPDATE & RESET
    # ==================
    qc.barrier(label='UPD')
    qc.cx(data[0], hidden[0])
    qc.cswap(hidden[0], hidden[1], anc[0])

    # Rot on hidden[0]: params[32:35]
    qc.rz(params[32], hidden[0])
    qc.ry(params[33], hidden[0])
    qc.rz(params[34], hidden[0])

    qc.cswap(hidden[0], hidden[3], anc[1])
    # ancilla 保持 |0⟩，不需 reset

    # ==================
    # 3. VARIATIONAL LAYERS（4 層）
    # ==================
    idx = 35
    for layer in range(4):
        qc.barrier(label=f'VAR{layer+1}')
        # Rot on all non-ancilla qubits
        for reg in [data, hidden]:
            for i in range(4):
                qc.rz(params[idx],     reg[i]); idx += 1
                qc.ry(params[idx],     reg[i]); idx += 1
                qc.rz(params[idx],     reg[i]); idx += 1

        # 循環 CX on all non-ancilla: data[0..3] → hidden[0..3] → back
        all_q = [data[i] for i in range(4)] + [hidden[i] for i in range(4)]
        for i in range(8):
            qc.cx(all_q[i], all_q[(i+1) % 8])

    return qc

In [ ]:
# ── Cell 3: OTC parameters and observables (from Comparsion.ipynb) ──
# ── 設定 ──────────────────────────────────────────────────────
CALIB_CSV  = '../error_mitigation_comparison/qru_otc_multi_obs_20260503_205659.csv'
OUT_CSV = None
N_REPS     = 3      # 每組電路重複 3 次
N_SHOTS    = 4000

# ── Observable 分組 ───────────────────────────────────────────
# (qubit, pauli, obs_name, out_idx, group_id, target_val)
OBS_GROUPS = {
    0: {  # Group 1: target = +0.8
        'target': +0.8,
        'obs': [
            (0, 'Z', 'Z0',  0),
            (4, 'Z', 'Z4',  8),
            (6, 'Z', 'Z6', 12),
            (5, 'X', 'X5', 11),
            (7, 'X', 'X7', 15),
        ]
    },
    1: {  # Group 2: target = -0.8
        'target': -0.8,
        'obs': [
            (1, 'Z', 'Z1',  2),
            (5, 'Z', 'Z5', 10),
            (7, 'Z', 'Z7', 14),
            (4, 'X', 'X4',  9),
            (6, 'X', 'X6', 13),
        ]
    },
}

# ── OTC 資料庫讀取 ────────────────────────────────────────────
param_cols = [c for c in pd.read_csv(CALIB_CSV, nrows=0).columns if c.startswith('p')]
calib_df   = pd.read_csv(CALIB_CSV)

def get_params(group_id, target_val, rep_id):
    """從 OTC 資料庫取得電路參數"""
    obs_name = OBS_GROUPS[group_id]['obs'][0][2]  # 用第一個 obs 查詢
    mask = (
        (calib_df['group_id'] == group_id) &
        (calib_df['target_val'] == target_val) &
        (calib_df['rep_id'] == rep_id) &
        (calib_df['obs_name'] == obs_name)
    )
    rows = calib_df[mask]
    if len(rows) == 0:
        return None, None
    row     = rows.iloc[0]
    params  = row[param_cols].values.astype(float)
    sv_vals = {}
    for _, r in calib_df[(calib_df['group_id']==group_id) &
                          (calib_df['target_val']==target_val) &
                          (calib_df['rep_id']==rep_id)].iterrows():
        sv_vals[r['obs_name']] = r['sv_val']
    return params, sv_vals

# ── Observable 建構 ───────────────────────────────────────────
def make_obs_list(group_id, n=10):
    obs_list = []
    for q, p, name, _ in OBS_GROUPS[group_id]['obs']:
        s = ['I'] * n
        s[n-1-q] = p
        obs_list.append(SparsePauliOp(''.join(s)))
    return obs_list


RESULTS_CSV = f'pec_repeats_results_{BACKEND_NAME}.csv'
def status_name(job):
    s = job.status()
    return s if isinstance(s, str) else s.name
def cz_error_sum(target, circ):
    return float(sum(target[ins.operation.name][tuple(circ.find_bit(q).index for q in ins.qubits)].error
                     for ins in circ.data if ins.operation.num_qubits == 2 and ins.operation.name != 'barrier'))

In [ ]:
# ── Cell 4: backend, transpile N_CIRCUITS distinct circuits, verification (no QPU) ──
if DRY_RUN:
    backend = AerSimulator.from_backend(FakeMarrakesh()); calib = FakeMarrakesh().target
else:
    service = QiskitRuntimeService(name=SERVICE_NAME, instance=INSTANCE)
    backend = service.backend(BACKEND_NAME); calib = backend.target
target_val = OBS_GROUPS[GRP_ID]['target']
obs_names  = [o[2] for o in OBS_GROUPS[GRP_ID]['obs']]
params, sv_vals = get_params(GRP_ID, target_val, REP)
sv_arr = np.array([sv_vals[n] for n in obs_names])
logical_qc = build_qru_circuit(np.zeros(8), np.zeros(8), params)

def target_without_couplers(T, couplers):
    new = Target(num_qubits=T.num_qubits, dt=T.dt, granularity=T.granularity, min_length=T.min_length,
                 pulse_alignment=T.pulse_alignment, acquire_alignment=T.acquire_alignment,
                 concurrent_measurements=T.concurrent_measurements)
    for name in T.operation_names:
        props = T[name]
        if props is None or None in props:
            new.add_instruction(T.operation_from_name(name), name=name); continue
        keep = {qa: p for qa, p in props.items() if not (len(qa) == 2 and (qa in couplers or qa[::-1] in couplers))}
        new.add_instruction(T.operation_from_name(name), keep, name=name)
    return new

def reduce_to_active(circ, ops):
    act = {circ.find_bit(q).index for ins in circ.data
           if ins.operation.name not in ('barrier', 'measure', 'delay', 'reset') for q in ins.qubits}
    for op in ops:
        act |= set(np.nonzero(np.any(op.paulis.z | op.paulis.x, axis=0))[0].tolist())
    act = sorted(act); m = {q: i for i, q in enumerate(act)}
    small = QuantumCircuit(len(act))
    for ins in circ.data:
        if ins.operation.name not in ('barrier', 'measure', 'delay', 'reset'):
            small.append(ins.operation, [m[circ.find_bit(q).index] for q in ins.qubits])
    return small, [SparsePauliOp(PauliList.from_symplectic(op.paulis.z[:, act], op.paulis.x[:, act]), op.coeffs)
                   for op in ops]

def cz_errors(target, circ):
    return [target[ins.operation.name][tuple(circ.find_bit(q).index for q in ins.qubits)].error
            for ins in circ.data if ins.operation.num_qubits == 2 and ins.operation.name != 'barrier']

label = time.strftime('%Y%m%d_%H%M%S')          # label of this batch (used by Cells 4b-7)
transpiled_at = datetime.now(timezone.utc).isoformat(timespec='seconds')
excluded = {qa for qa, p in calib['cz'].items() if p is not None and p.error is not None and p.error > CZ_ERR_MAX}
reduced_target = target_without_couplers(calib, excluded)

circuits, skipped = [], []                       # circuits: list of dicts (one per distinct circuit)
for seed in range(FIRST_SEED, MAX_SEED + 1):
    if len(circuits) == N_CIRCUITS:
        break
    pm = generate_preset_pass_manager(target=reduced_target, optimization_level=1, seed_transpiler=seed)
    qc = pm.run(logical_qc)
    if any(qc == c['isa_qc'] for c in circuits):  # identical to a circuit already kept
        skipped.append(seed); continue
    obs = [ob.apply_layout(qc.layout) for ob in make_obs_list(GRP_ID)]
    small, small_ops = reduce_to_active(qc, obs)
    dev = np.max(np.abs(np.asarray(StatevectorEstimator().run([(small, small_ops)]).result()[0].data.evs) - sv_arr))
    assert dev < 1e-6, f"seed {seed}: circuit does not reproduce the SV reference ({dev:.1e})"
    errs = cz_errors(calib, qc)
    tag = f'{BACKEND_NAME}_seed{seed}_excl{int(CZ_ERR_MAX*100)}pct_{label}'
    path = f'pec_isa_circuit_{tag}.qpy'
    qpy.dump(qc, open(path, 'wb'))
    json.dump({n: op.to_list() for n, op in zip(obs_names, obs)},
              open(f'pec_observables_{tag}.json', 'w'), default=str)
    circuits.append({'seed': seed, 'circuit_file': path, 'isa_qc': qc, 'isa_obs': obs, 'sv_check': float(dev),
                     'depth': qc.depth(), 'depth_2q': qc.depth(lambda ins: ins.operation.num_qubits == 2),
                     'n_cz': len(errs), 'sum_cz_error': float(sum(errs)), 'max_cz_error': float(max(errs)),
                     'physical_qubits': [int(q) for q in qc.layout.final_index_layout()[:logical_qc.num_qubits]],
                     'active_qubits': small.num_qubits})
assert len(circuits) == N_CIRCUITS, f"only {len(circuits)} distinct circuits up to seed {MAX_SEED}"

KEYS = ['seed', 'circuit_file', 'sv_check', 'depth', 'depth_2q', 'n_cz', 'sum_cz_error', 'max_cz_error',
        'physical_qubits', 'active_qubits']
summary = {'label': label, 'backend': BACKEND_NAME, 'transpiled_utc': transpiled_at,
           'rule': f'optimization_level=1, CZ couplers with reported error > {CZ_ERR_MAX} excluded, seeds from {FIRST_SEED} upward, identical circuits skipped',
           'n_directed_couplers_excluded': len(excluded), 'seeds_skipped_identical': skipped,
           'group_id': GRP_ID, 'rep_id': REP, 'observables': obs_names,
           'circuits': [{k: c[k] for k in KEYS} for c in circuits]}
json.dump(summary, open(f'pec_repeats_circuits_{BACKEND_NAME}_{label}.json', 'w'), indent=2)

print(f"{backend.name} | batch {label} | {len(excluded)} directed couplers excluded | "
      f"seeds skipped (identical): {skipped or 'none'}")
print(pd.DataFrame([{k: c[k] for k in KEYS if k not in ('circuit_file',)} for c in circuits])
      .to_string(index=False, float_format=lambda v: f'{v:.3g}'))
print(f"saved {N_CIRCUITS} QPY + observables, and pec_repeats_circuits_{BACKEND_NAME}_{label}.json")

In [ ]:
# ── Cell 4b: circuit drawings (no QPU) ──
# 'mpl' needs pylatexenc (pip install pylatexenc); falls back to text drawings if it is missing
from IPython.display import display
try:
    import pylatexenc; DRAW = 'mpl'
except ImportError:
    DRAW = 'text'; print("pylatexenc not installed -> text drawings")
names = [f'{r.name}{i}' for r in logical_qc.qregs for i in range(r.size)]
print("logical circuit (10 qubits): compare with the QRU circuit figure in the paper")
display(logical_qc.draw(DRAW, fold=40))
for c in circuits:
    qc = c['isa_qc']
    print(f"\n── seed {c['seed']}: depth {c['depth']}, 2q depth {c['depth_2q']}, CZ {c['n_cz']}, "
          f"summed CZ error {c['sum_cz_error']:.3f}, {c['active_qubits']} active qubits ──")
    print("logical -> physical:", ", ".join(f"{n}->{p}" for n, p in zip(names, c['physical_qubits'])))
    print("gate counts:", dict(qc.count_ops()))
    display(qc.draw(DRAW, idle_wires=False, fold=80, scale=0.5))

In [ ]:
# ── Cell 5: options, estimate and budget check (no QPU) ──
opts = EstimatorOptions()
opts.resilience_level = 1
opts.resilience.measure_mitigation = True
opts.resilience.pec_mitigation = not DRY_RUN
opts.resilience.layer_noise_learning.max_layers_to_learn = None
opts.resilience.pec.max_overhead = None
if PEC_PRECISION is not None:
    opts.default_precision = PEC_PRECISION
if not DRY_RUN:
    opts.max_execution_time = MAX_EXEC_S
a, b, r = LN_FIT
rows = []
for c in circuits:
    ln_mid = a * c['sum_cz_error'] + b
    rows.append({'seed': c['seed'], 'sum_cz_error': c['sum_cz_error'],
                 'overhead_est': f"{np.exp(ln_mid - r):.0f}-{np.exp(ln_mid + r):.0f}",
                 'overhead_hi': np.exp(ln_mid + r)})
est_df = pd.DataFrame(rows)
print(est_df.drop(columns='overhead_hi').to_string(index=False, float_format=lambda v: f'{v:.3f}'))
need_s = N_CIRCUITS * N_REPEATS * SEC_PER_RUN
print(f"\nplanning: {N_CIRCUITS} circuits x {N_REPEATS} runs x ~{SEC_PER_RUN} s = ~{need_s/60:.0f} min "
      f"(valid only if the overheads are ~10, as on 9 Oct) | budget {QPU_BUDGET_S/60:.0f} min -> "
      f"{'OK' if need_s <= QPU_BUDGET_S else 'EXCEEDS'}")
if est_df['overhead_hi'].max() > 30:
    print("WARNING: some estimated overheads are well above the 9 Oct value (10); "
          "runs will take longer than planned — check before submitting.")

In [ ]:
# ── Cell 6: SUBMIT all runs in one batch (returns immediately) ──
# order: interleaved (seed A, B, C, D, A, B, ...) so that slow drift during the batch is not tied to one circuit
order = [(rep, c) for rep in range(N_REPEATS) for c in circuits]
jobs = []
if DRY_RUN:
    est = Estimator(mode=backend, options=opts)
    jobs_local = {}
    for rep, c in order:
        j = est.run([(c['isa_qc'], c['isa_obs'])]); jobs_local[j.job_id()] = j
        jobs.append({'seed': c['seed'], 'repeat': rep, 'circuit_file': c['circuit_file'], 'job_id': j.job_id()})
else:
    Mode = Batch if EXEC_MODE == "batch" else Session
    with Mode(backend=backend) as mode:
        est = Estimator(mode=mode, options=opts)
        for rep, c in order:
            jid = est.run([(c['isa_qc'], c['isa_obs'])]).job_id()
            jobs.append({'seed': c['seed'], 'repeat': rep, 'circuit_file': c['circuit_file'], 'job_id': jid})
pending = {'label': label, 'backend': BACKEND_NAME, 'instance': INSTANCE, 'exec_mode': EXEC_MODE,
           'submitted': time.strftime('%Y-%m-%d %H:%M:%S'),
           'circuits_file': f'pec_repeats_circuits_{BACKEND_NAME}_{label}.json', 'jobs': jobs}
with open(f'pec_repeats_pending_{BACKEND_NAME}_{label}.json', 'w') as f:       # local only (has job ids)
    json.dump(pending, f, indent=2)
with open('pec_job_ids.txt', 'a') as f:
    for j in jobs:
        f.write(f"{pending['submitted']} {BACKEND_NAME} repeat_{label} seed{j['seed']} rep{j['repeat']} {j['job_id']}\n")
print(f"submitted {len(jobs)} jobs ({N_CIRCUITS} circuits x {N_REPEATS}) in one {EXEC_MODE}; "
      f"saved pec_repeats_pending_{BACKEND_NAME}_{label}.json")

In [ ]:
# ── Cell 7: COLLECT — re-run until all jobs are DONE (safe after a kernel restart: run Cells 1-3 first) ──
def parse_ts(s):
    if s is None: return None
    return datetime.fromisoformat(str(s).replace('Z', '+00:00'))
obs_names = [o[2] for o in OBS_GROUPS[GRP_ID]['obs']]
_, sv_vals = get_params(GRP_ID, OBS_GROUPS[GRP_ID]['target'], REP)
sv_arr = np.array([sv_vals[n] for n in obs_names])
done = pd.read_csv(RESULTS_CSV) if os.path.exists(RESULTS_CSV) else pd.DataFrame()
done_keys = set(zip(done['batch'].astype(str), done['seed'], done['repeat'])) if len(done) else set()
if not DRY_RUN:
    svc = QiskitRuntimeService(name=SERVICE_NAME, instance=INSTANCE)
    bk = svc.backend(BACKEND_NAME)
new_rows, counts, circ_cache = [], {}, {}
for pf in sorted(glob.glob(f'pec_repeats_pending_{BACKEND_NAME}_*.json')):
    pend = json.load(open(pf))
    if 'jobs' not in pend:                       # pending file of an older notebook version
        continue
    info = {c['seed']: c for c in json.load(open(pend['circuits_file']))['circuits']}
    for jd in pend['jobs']:
        key = (str(pend['label']), jd['seed'], jd['repeat'])
        if key in done_keys:
            counts['saved'] = counts.get('saved', 0) + 1; continue
        job = jobs_local[jd['job_id']] if DRY_RUN else svc.job(jd['job_id'])
        st = status_name(job); counts[st] = counts.get(st, 0) + 1
        if st != 'DONE':
            continue
        if jd['circuit_file'] not in circ_cache:
            circ_cache[jd['circuit_file']] = qpy.load(open(jd['circuit_file'], 'rb'))[0]
        circ = circ_cache[jd['circuit_file']]
        res = job.result(); meta = res[0].metadata
        evs, stds = np.asarray(res[0].data.evs), np.asarray(res[0].data.stds)
        rel, z = (evs - sv_arr) / sv_arr, (evs - sv_arr) / stds
        ln_ = meta.get('resilience', {}).get('layer_noise', {})
        per_layer = ln_.get('unique_mitigated_layers_noise_overhead') or []
        t_run, cz_run, qpu = None, np.nan, np.nan
        if not DRY_RUN:
            qpu = job.usage()
            t_run = parse_ts((job.metrics().get('timestamps') or {}).get('running'))
            if t_run is not None:
                cz_run = cz_error_sum(bk.target_history(datetime=t_run), circ)
        row = {'batch': str(pend['label']), 'seed': jd['seed'], 'repeat': jd['repeat'],
               'exec_mode': pend.get('exec_mode'), 'n_cz': info[jd['seed']]['n_cz'],
               'depth': info[jd['seed']]['depth'], 'started_utc': t_run.isoformat() if t_run else None,
               'sum_cz_error_at_start': cz_run, 'sum_cz_error_at_transpile': info[jd['seed']]['sum_cz_error'],
               'noise_overhead': ln_.get('noise_overhead'), 'mean_ln_layer_overhead':
               (float(np.mean(np.log(per_layer))) if per_layer else np.nan),
               'n_layers_total': ln_.get('total_mitigated_layers'), 'n_layers_unique': ln_.get('unique_mitigated_layers'),
               'shots': meta.get('shots'), 'qpu_s': qpu, 'mean_dev_pct': 100 * rel.mean(),
               'min_dev_pct': 100 * rel.min(), 'max_dev_pct': 100 * rel.max(), 'max_abs_z': float(np.max(np.abs(z))),
               'mean_std': float(stds.mean())}
        row.update({f'ev_{n}': float(e) for n, e in zip(obs_names, evs)})
        row.update({f'std_{n}': float(s) for n, s in zip(obs_names, stds)})
        new_rows.append(row)
if new_rows:
    done = pd.concat([done, pd.DataFrame(new_rows)], ignore_index=True)
    done.to_csv(RESULTS_CSV, index=False)        # no job ids in this file
print("status:", counts, f"| {len(new_rows)} new results saved to {RESULTS_CSV} ({len(done)} total)")
if len(done):
    print(done[['batch', 'seed', 'repeat', 'sum_cz_error_at_start', 'noise_overhead', 'mean_dev_pct',
                'min_dev_pct', 'max_dev_pct', 'qpu_s']].sort_values(['batch', 'seed', 'repeat'])
          .to_string(index=False, float_format=lambda v: f'{v:.3g}'))

In [ ]:
# ── Cell 7b: execution times (no QPU) ──
t_rows = []
for pf in sorted(glob.glob(f'pec_repeats_pending_{BACKEND_NAME}_*.json')):
    pend = json.load(open(pf))
    for jd in pend.get('jobs', []):
        ts = (svc.job(jd['job_id']).metrics().get('timestamps') or {})
        t_rows.append({'batch': str(pend['label']), 'seed': jd['seed'], 'repeat': jd['repeat'],
                       'created': parse_ts(ts.get('created')), 'running': parse_ts(ts.get('running')),
                       'finished': parse_ts(ts.get('finished'))})
tdf = pd.DataFrame(t_rows).sort_values('running').reset_index(drop=True)
tdf['order'] = range(len(tdf))
tdf['wall_s'] = (tdf['finished'] - tdf['running']).dt.total_seconds()
t0 = tdf['running'].min()
tdf['start_min'] = (tdf['running'] - t0).dt.total_seconds() / 60
tdf['end_min'] = (tdf['finished'] - t0).dt.total_seconds() / 60
res = pd.read_csv(RESULTS_CSV); res['batch'] = res['batch'].astype(str)
tdf = tdf.merge(res[['batch', 'seed', 'repeat', 'mean_dev_pct', 'qpu_s']], on=['batch', 'seed', 'repeat'])
print(tdf[['order', 'seed', 'repeat', 'start_min', 'end_min', 'wall_s', 'qpu_s', 'mean_dev_pct']]
      .to_string(index=False, float_format=lambda v: f'{v:.2f}'))
overlap = (tdf['running'].iloc[1:].values < tdf['finished'].iloc[:-1].values).sum()
print(f"\nconsecutive jobs overlapping in time: {overlap} of {len(tdf)-1}")
print(f"correlation of mean deviation with execution order: r = {np.corrcoef(tdf['order'], tdf['mean_dev_pct'])[0, 1]:+.2f}")

In [ ]:
# ── Cell 7c: summed CZ error at job end vs job start (no QPU) ──
circ_cache = {}
rows = []
for pf in sorted(glob.glob(f'pec_repeats_pending_{BACKEND_NAME}_*.json')):
    pend = json.load(open(pf))
    for jd in pend.get('jobs', []):
        if jd['circuit_file'] not in circ_cache:
            circ_cache[jd['circuit_file']] = qpy.load(open(jd['circuit_file'], 'rb'))[0]
        circ = circ_cache[jd['circuit_file']]
        r = tdf[(tdf['seed'] == jd['seed']) & (tdf['repeat'] == jd['repeat'])].iloc[0]
        rows.append({'seed': jd['seed'], 'repeat': jd['repeat'],
                     'cz_start': cz_error_sum(bk.target_history(datetime=r['running']), circ),
                     'cz_end': cz_error_sum(bk.target_history(datetime=r['finished']), circ)})
czdf = pd.DataFrame(rows).sort_values(['seed', 'repeat'])
czdf['change'] = czdf['cz_end'] - czdf['cz_start']
print(czdf.to_string(index=False, float_format=lambda v: f'{v:.4f}'))
print("\ncalibration snapshot at end differs from start for", int((czdf['change'].abs() > 1e-9).sum()), "of", len(czdf), "jobs")

In [ ]:
# ── Cell 8: analysis ──
import matplotlib.pyplot as plt
df = pd.read_csv(RESULTS_CSV).copy()
df['cz'] = df['sum_cz_error_at_start'].fillna(df['sum_cz_error_at_transpile'])
df['ln_overhead'] = np.log(df['noise_overhead'].astype(float))
df['circuit'] = df['batch'].astype(str) + '/s' + df['seed'].astype(str)
ev_cols, sd_cols = [f'ev_{n}' for n in obs_names], [f'std_{n}' for n in obs_names]

# 1) per circuit (reproducibility within the same ISA circuit)
g = df.groupby('circuit').agg(n=('mean_dev_pct', 'size'), cz=('cz', 'mean'), cz_range=('cz', lambda v: v.max() - v.min()),
                              overhead=('noise_overhead', 'mean'), mean_dev=('mean_dev_pct', 'mean'),
                              sd_dev=('mean_dev_pct', 'std'), lo=('mean_dev_pct', 'min'), hi=('mean_dev_pct', 'max'))
print("per circuit (mean deviation over the five observables, %):\n" + g.to_string(float_format=lambda v: f'{v:.3g}'))

# 2) within-circuit spread of each observable vs the reported uncertainties
#    chi2/dof ≈ 1: repeats agree within their reported stds; >> 1: they do not
chi2, dof = 0.0, 0
for _, d in df.groupby('circuit'):
    if len(d) < 2: continue
    e, s = d[ev_cols].to_numpy(), d[sd_cols].to_numpy()
    w = 1 / s**2; mean_w = (w * e).sum(0) / w.sum(0)
    chi2 += float((((e - mean_w) / s)**2).sum()); dof += e.size - e.shape[1]
if dof:
    print(f"\nwithin-circuit consistency of the observables: chi2/dof = {chi2/dof:.2f} (dof {dof})")

# 3) within vs between circuits, and dependence on noise
within_sd = np.sqrt((df.groupby('circuit')['mean_dev_pct'].var(ddof=1).dropna()).mean())
between_sd = g['mean_dev'].std()
print(f"pooled within-circuit SD of the mean deviation: {within_sd:.1f}% | SD of the circuit means: {between_sd:.1f}%")
print(f"expected SD from the reported uncertainties alone: ~{100 * (df['mean_std'] / np.sqrt(5) / np.abs(sv_arr).mean()).mean():.1f}% "
      "(if the 5 observables were independent)")
print(f"all {len(df)} runs: mean deviation average {df['mean_dev_pct'].mean():+.1f}%, "
      f"range {df['mean_dev_pct'].min():+.1f}% to {df['mean_dev_pct'].max():+.1f}%")
for xcol, name in [('cz', 'summed CZ error at job start'), ('ln_overhead', 'ln(learned overhead)'),
                   ('mean_ln_layer_overhead', 'mean ln per-layer overhead')]:
    ok = df[[xcol, 'mean_dev_pct']].dropna()
    if len(ok) >= 3 and ok[xcol].std() > 1e-6:
        print(f"correlation of mean deviation with {name}: r = {np.corrcoef(ok[xcol], ok['mean_dev_pct'])[0, 1]:+.2f} (n = {len(ok)})")
ok = df[['cz', 'ln_overhead']].dropna()
if len(ok) >= 3 and ok['cz'].std() > 1e-6:
    k, c0 = np.polyfit(ok['cz'], ok['ln_overhead'], 1)
    print(f"ln(overhead) vs summed CZ error in these runs: slope {k:.1f}, intercept {c0:+.2f} (earlier fit: {LN_FIT[0]}, {LN_FIT[1]})")

# 4) earlier single run (9 Oct, its own transpile) for reference
for f in INCLUDE_PREVIOUS:
    if os.path.exists(f):
        r = json.load(open(f)); e = np.array(r['evs']); rel = (e - sv_arr) / sv_arr
        oh = r['pub_metadata'].get('resilience', {}).get('layer_noise', {}).get('noise_overhead')
        print(f"\nearlier run {f}: summed CZ error {r['circuit']['sum_cz_error']:.3f}, overhead {oh}, "
              f"mean deviation {100*rel.mean():+.1f}% ({100*rel.min():+.1f}% to {100*rel.max():+.1f}%)")

# 5) plots
fig, ax = plt.subplots(1, 3, figsize=(15, 4.2))
for i, (cid, d) in enumerate(df.groupby('circuit')):
    ax[0].plot(d['cz'], d['mean_dev_pct'], 'o', color=f'C{i}', label=cid)
    ax[1].plot(d['ln_overhead'], d['mean_dev_pct'], 'o', color=f'C{i}', label=cid)
for a, xl in zip(ax[:2], ['summed CZ error (calibration at job start)', 'ln(sampling overhead, learned)']):
    a.axhline(0, color='0.6', lw=0.8); a.set_xlabel(xl); a.set_ylabel('mean deviation from SV (%)')
ax[0].legend(fontsize=8)
circ_ids = sorted(df['circuit'].unique()); width = 0.8 / max(len(df), 1); xpos = np.arange(len(obs_names)); k = 0
for i, cid in enumerate(circ_ids):
    for _, row in df[df['circuit'] == cid].sort_values('repeat').iterrows():
        ax[2].errorbar(xpos - 0.4 + (k + 0.5) * width, row[ev_cols].to_numpy(float), yerr=row[sd_cols].to_numpy(float),
                       fmt='o', ms=3, color=f'C{i}', elinewidth=0.8); k += 1
ax[2].plot(xpos, sv_arr, '_', ms=30, color='k', label='SV'); ax[2].set_xticks(xpos, obs_names)
ax[2].set_ylabel('expectation value'); ax[2].legend(fontsize=8)
fig.suptitle(f'Repeated full PEC on {BACKEND_NAME}: {df["circuit"].nunique()} circuits'); fig.tight_layout(); plt.show()